# Paper 1 demo: Ball-DP ridge prototypes on MNIST embeddings

This notebook is a **single-dataset end-to-end tutorial** for the Paper 1 convex release and finite-prior attack API. It is not the paper-scale experiment runner. The official scripts sweep datasets, radii, privacy targets, support sizes, and seeds; this notebook shows the core workflow once.

We will:

1. load MNIST embeddings;
2. estimate a label-preserving Ball radius;
3. train a Ball-DP ridge-prototype release and a standard-DP comparator;
4. compute exact-identification upper bounds with `ball_rero`;
5. build one finite Ball-local candidate support;
6. run the exact finite-prior MAP attack against both releases.


## Mathematical setup

A record is an embedding-label pair
\[
z=(x,y),\qquad x\in\mathbb R^d.
\]
The label-preserving metric is
\[
d((x,y),(x',y'))=
\begin{cases}
\|x-x'\|_2,& y=y',\\
\infty,& y\ne y'.
\end{cases}
\]
Ball-DP restricts neighboring replacements to records satisfying \(d(z,z')\le r\). The ridge-prototype model solves
\[
\min_{\mu_1,\ldots,\mu_K}
\frac{1}{n}\sum_{i=1}^n \|x_i-\mu_{y_i}\|_2^2
+\frac{\lambda}{2}\sum_{c=1}^K\|\mu_c\|_2^2.
\]
The released model is Gaussian output perturbation,
\[
\widetilde\mu=\hat\mu(D)+\xi,
\qquad \xi\sim\mathcal N(0,\sigma^2I).
\]
For exact identification over a uniform finite prior of size \(m\), the oblivious baseline is \(\kappa=1/m\). The direct Gaussian ReRo bound has the form
\[
\gamma
=\Phi\left(\Phi^{-1}(\kappa)+\frac{\Delta}{\sigma}\right),
\]
where \(\Delta\) is the relevant model-output sensitivity.


In [ ]:
from __future__ import annotations

from types import SimpleNamespace
import json
import math
import sys
from pathlib import Path

import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "quantbayes").exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quantbayes.ball_dp import (
    attack_convex_ball_output_finite_prior,
    ball_rero,
    diagnose_convex_ball_output_finite_prior,
    fit_convex,
    make_finite_identification_prior,
    summarize_embedding_ball_radii,
)
from quantbayes.ball_dp.experiments.run_attack_experiment import (
    DEFAULT_ORDERS,
    LoadedDataset,
    actual_ball_epsilon,
    actual_standard_epsilon_same_noise,
    find_feasible_support_banks,
    load_embeddings,
    make_support_set,
    radius_value_from_report,
    remove_train_index,
    resolve_dataset,
    standard_radius_value_from_report,
    validate_model_dataset_compatibility,
)

plt.rcParams.update({
    "figure.figsize": (7.2, 4.4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
})


## Configuration

These defaults keep the notebook small enough for interactive use. Increase the subset sizes or number of seeds in the official scripts, not here.


In [ ]:
DATASET = "MNIST-embeddings"
MODEL_FAMILY = "ridge_prototype"

MAX_TRAIN_EXAMPLES = 4000
MAX_TEST_EXAMPLES = 1000
SUBSAMPLE_SEED = 0

RADIUS_TAG = "q80"
EPSILON = 8.0
DELTA = 1e-6
M = 8
LAMBDA = 1e-2
RELEASE_SEED = 0
EMBEDDING_BOUND = 1.0
RIDGE_SENSITIVITY_MODE = "count_aware"
STANDARD_RADIUS_SOURCE = "empirical_diameter"

SUPPORT_SOURCE = "public_only"
SUPPORT_SELECTION = "farthest"
ANCHOR_SELECTION = "rare_class"
MAX_FEASIBLE_SEARCH = 5000

loader_args = SimpleNamespace(
    data_root="./data",
    embedding_batch_size=128,
    allow_cpu_fallback=True,
    embedding_cache_path=None,
    force_recompute_embeddings=False,
    num_workers=2,
    hf_cache_dir=None,
    encoder_model_name="sentence-transformers/all-MiniLM-L6-v2",
    max_length=256,
)

rng = np.random.default_rng(0)


In [ ]:
def stratified_subsample(X: np.ndarray, y: np.ndarray, max_examples: int | None, *, seed: int):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int32).reshape(-1)
    if max_examples is None or int(max_examples) <= 0 or int(max_examples) >= len(y):
        return X, y

    rng = np.random.default_rng(seed)
    labels, counts = np.unique(y, return_counts=True)
    if int(max_examples) < len(labels):
        idx = rng.choice(np.arange(len(y)), size=int(max_examples), replace=False)
        return X[idx], y[idx]

    raw = counts / counts.sum() * int(max_examples)
    alloc = np.maximum(1, np.floor(raw).astype(int))
    alloc = np.minimum(alloc, counts)
    while int(alloc.sum()) > int(max_examples):
        j = int(np.argmax(alloc))
        if alloc[j] > 1:
            alloc[j] -= 1
        else:
            break
    while int(alloc.sum()) < int(max_examples):
        room = counts - alloc
        candidates = np.flatnonzero(room > 0)
        if candidates.size == 0:
            break
        j = int(candidates[np.argmax(room[candidates])])
        alloc[j] += 1

    keep: list[int] = []
    for label, take in zip(labels, alloc, strict=True):
        idx = np.flatnonzero(y == label)
        keep.extend(rng.choice(idx, size=int(take), replace=False).tolist())
    rng.shuffle(keep)
    idx = np.asarray(keep, dtype=np.int64)
    return X[idx], y[idx]


## Load MNIST embeddings

The loader caches embeddings. The first run can be slow if the cache is absent; later runs reuse the cached embedding arrays.


In [ ]:
spec = resolve_dataset(DATASET)
loaded = load_embeddings(loader_args, spec)

X_train, y_train = stratified_subsample(
    loaded.X_train, loaded.y_train, MAX_TRAIN_EXAMPLES, seed=SUBSAMPLE_SEED
)
X_test, y_test = stratified_subsample(
    loaded.X_test, loaded.y_test, MAX_TEST_EXAMPLES, seed=SUBSAMPLE_SEED + 17
)

demo_data = LoadedDataset(
    spec=loaded.spec,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    label_values=loaded.label_values,
    num_classes=int(len(np.unique(y_train))),
    feature_dim=int(X_train.shape[1]),
    empirical_embedding_bound=float(np.max(np.linalg.norm(X_train, axis=1))),
    backend=loaded.backend,
)
validate_model_dataset_compatibility(MODEL_FAMILY, demo_data)

if demo_data.empirical_embedding_bound > EMBEDDING_BOUND + 1e-3:
    raise ValueError(
        f"Empirical embedding bound {demo_data.empirical_embedding_bound:.4f} exceeds "
        f"public EMBEDDING_BOUND={EMBEDDING_BOUND}. Increase EMBEDDING_BOUND."
    )

pd.DataFrame([
    {
        "dataset": spec.display_name,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "num_classes": demo_data.num_classes,
        "feature_dim": demo_data.feature_dim,
        "empirical_embedding_bound": demo_data.empirical_embedding_bound,
        "public_embedding_bound": EMBEDDING_BOUND,
        "backend": demo_data.backend,
    }
])


## Estimate Ball and standard comparator radii

For the Ball mechanism, we use the worst labelwise quantile radius
\[
r_{\mathrm{Ball}}=\max_c \widehat Q_{0.80}\{\|x_i-x_j\|_2:y_i=y_j=c\}.
\]
For the standard comparator, we use the same-label empirical diameter unless configured otherwise:
\[
r_{\mathrm{Std}}=\max_c\max_{i,j:y_i=y_j=c}\|x_i-x_j\|_2.
\]


In [ ]:
radius_report = summarize_embedding_ball_radii(
    X_train,
    y_train,
    quantiles=(0.50, 0.80, 0.95),
    max_exact_pairs=250_000,
    max_sampled_pairs=100_000,
    seed=0,
)
r_ball = radius_value_from_report(radius_report, RADIUS_TAG)
r_std = standard_radius_value_from_report(
    radius_report,
    source=STANDARD_RADIUS_SOURCE,
    embedding_bound=EMBEDDING_BOUND,
)

radius_df = pd.DataFrame([
    {"quantity": "Ball radius", "tag/source": RADIUS_TAG, "value": r_ball},
    {"quantity": "Standard radius", "tag/source": STANDARD_RADIUS_SOURCE, "value": r_std},
    {"quantity": "Public bound comparator", "tag/source": "2B", "value": 2.0 * EMBEDDING_BOUND},
    {"quantity": "Std/Ball ratio", "tag/source": "ratio", "value": r_std / r_ball},
])
display(radius_df)


## Train one Ball release and one standard comparator

The two calls use the same model, regularization, privacy target, and seed. The only intended difference is the release radius used to calibrate the Gaussian perturbation.


In [ ]:
def fit_ridge_release(*, mechanism: str, radius: float):
    return fit_convex(
        X_train,
        y_train,
        X_eval=X_test,
        y_eval=y_test,
        model_family=MODEL_FAMILY,
        privacy="ball_dp",
        radius=float(radius),
        standard_radius=float(r_std),
        ridge_sensitivity_mode=RIDGE_SENSITIVITY_MODE,
        lam=LAMBDA,
        epsilon=EPSILON,
        delta=DELTA,
        embedding_bound=EMBEDDING_BOUND,
        num_classes=demo_data.num_classes,
        orders=DEFAULT_ORDERS,
        max_iter=100,
        solver="lbfgs_fullbatch",
        seed=RELEASE_SEED,
    )

ball_release = fit_ridge_release(mechanism="ball", radius=r_ball)
standard_release = fit_ridge_release(mechanism="standard", radius=r_std)

def release_row(name: str, release):
    return {
        "mechanism": name,
        "accuracy": float(release.utility_metrics.get("accuracy", np.nan)),
        "sigma": float(release.privacy.ball.sigma),
        "sensitivity_delta": float(release.sensitivity.delta_ball),
        "release_radius": float(release.privacy.ball.radius),
        "actual_ball_epsilon": actual_ball_epsilon(release),
        "same_noise_standard_epsilon": actual_standard_epsilon_same_noise(release),
    }

release_df = pd.DataFrame([
    release_row("Ball-DP", ball_release),
    release_row("Standard DP", standard_release),
])
release_df["sigma_ratio_vs_ball"] = release_df["sigma"] / release_df.loc[0, "sigma"]
display(release_df)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.4), constrained_layout=True)
axes[0].bar(release_df["mechanism"], release_df["accuracy"])
axes[0].set_title("Accuracy")
axes[0].set_ylim(0, 1)

axes[1].bar(release_df["mechanism"], release_df["sigma"])
axes[1].set_title("Noise scale")
axes[1].set_ylabel(r"$\sigma$")

axes[2].bar(release_df["mechanism"], release_df["sensitivity_delta"])
axes[2].set_title("Sensitivity")
axes[2].set_ylabel(r"$\Delta_2$")
plt.show()


## Compute ReRo finite-prior bounds

The prior here is a dummy uniform finite prior of size \(m\). For exact-identification bounds, only \(m\) matters; the geometry of these dummy samples is irrelevant.


In [ ]:
finite_prior = make_finite_identification_prior(
    np.zeros((M, demo_data.feature_dim), dtype=np.float32),
    weights=None,
)

def bound_row(name: str, release):
    dp = ball_rero(release, prior=finite_prior, eta_grid=(0.5,), mode="dp")
    direct = ball_rero(release, prior=finite_prior, eta_grid=(0.5,), mode="gaussian_direct")
    rdp = ball_rero(release, prior=finite_prior, eta_grid=(0.5,), mode="rdp")
    return {
        "mechanism": name,
        "oblivious_baseline": float(direct.points[0].kappa),
        "generic_dp_bound": float(dp.points[0].gamma_ball),
        "direct_gaussian_bound": float(direct.points[0].gamma_ball),
        "rdp_bound": float(rdp.points[0].gamma_ball),
        "standard_same_noise_bound": (
            np.nan if direct.points[0].gamma_standard is None else float(direct.points[0].gamma_standard)
        ),
    }

bound_df = pd.DataFrame([
    bound_row("Ball-DP", ball_release),
    bound_row("Standard DP", standard_release),
])
display(bound_df)


In [ ]:
plot_cols = ["oblivious_baseline", "direct_gaussian_bound", "rdp_bound"]
bound_df.set_index("mechanism")[plot_cols].plot(kind="bar")
plt.ylabel("Exact-ID success upper bound")
plt.ylim(0, 1.02)
plt.title(f"Finite-prior bounds, m={M}, epsilon={EPSILON:g}")
plt.xticks(rotation=0)
plt.show()


## Build one Ball-local finite support

We now choose an anchor \(u\) from training data and construct a public/test support
\[
S=\{z_1,\ldots,z_m\}\subseteq \mathcal B(u,r).
\]
The anchor defines \(D^-=D\setminus\{u\}\). The hidden target is one candidate from \(S\), not the anchor itself. This matches the official attack script.


In [ ]:
feasible_banks = find_feasible_support_banks(
    demo_data,
    radius_value=float(r_ball),
    max_required_m=M,
    num_supports=1,
    anchor_seed=0,
    source_mode=SUPPORT_SOURCE,
    max_search=MAX_FEASIBLE_SEARCH,
    explicit_anchor_indices=None,
    strict=True,
    anchor_selection=ANCHOR_SELECTION,
)
bank = feasible_banks[0]
X_support, y_support, source_ids, support_dists = make_support_set(
    bank,
    m=M,
    draw_index=0,
    base_seed=0,
    selection=SUPPORT_SELECTION,
)

support_df = pd.DataFrame({
    "candidate": np.arange(M),
    "source_id": source_ids,
    "label": y_support,
    "distance_to_anchor": support_dists,
})
display(pd.DataFrame([
    {
        "anchor_index": bank.anchor_index,
        "anchor_label": bank.anchor_label,
        "support_size": M,
        "candidate_bank_size": int(bank.bank_vectors.shape[0]),
        "support_selection": SUPPORT_SELECTION,
        "support_max_distance": float(np.max(support_dists)),
    }
]))
display(support_df)


## Run the exact finite-prior MAP attack

For each target candidate \(z_i\in S\), we form
\[
D_i=D^-\cup\{z_i\},
\]
release a noisy model, and run the Bayes/MAP finite-prior attack over the same support. This cell trains \(2m\) small ridge releases: one Ball-DP and one standard comparator for each target.


In [ ]:
X_minus, y_minus = remove_train_index(X_train, y_train, bank.anchor_index)

attack_rows: list[dict] = []
for target_pos, target_sid in enumerate(source_ids):
    x_target = X_support[target_pos]
    y_target = int(y_support[target_pos])
    X_full = np.concatenate([X_minus, x_target[None, :]], axis=0).astype(np.float32)
    y_full = np.concatenate([y_minus, np.asarray([y_target], dtype=np.int32)], axis=0)
    target_index = len(X_full) - 1

    for mechanism, radius_value in [("Ball-DP", r_ball), ("Standard DP", r_std)]:
        release = fit_convex(
            X_full,
            y_full,
            X_eval=X_test,
            y_eval=y_test,
            model_family=MODEL_FAMILY,
            privacy="ball_dp",
            radius=float(radius_value),
            standard_radius=float(r_std),
            ridge_sensitivity_mode=RIDGE_SENSITIVITY_MODE,
            lam=LAMBDA,
            epsilon=EPSILON,
            delta=DELTA,
            embedding_bound=EMBEDDING_BOUND,
            num_classes=demo_data.num_classes,
            orders=DEFAULT_ORDERS,
            max_iter=100,
            solver="lbfgs_fullbatch",
            seed=RELEASE_SEED,
        )

        attack, _, _ = attack_convex_ball_output_finite_prior(
            release,
            X_full,
            y_full,
            target_index=target_index,
            X_candidates=X_support,
            y_candidates=y_support,
            prior_weights=None,
            known_label=int(bank.anchor_label),
            eta_grid=(0.5,),
        )
        diagnostics = diagnose_convex_ball_output_finite_prior(
            release,
            X_full,
            y_full,
            target_index=target_index,
            X_candidates=X_support,
            y_candidates=y_support,
            prior_weights=None,
            known_label=int(bank.anchor_label),
            center_features=np.asarray(bank.anchor_vector, dtype=np.float32),
            center_label=int(bank.anchor_label),
        )
        pred_idx = attack.diagnostics.get("predicted_prior_index")
        attack_rows.append({
            "mechanism": mechanism,
            "target_position": target_pos,
            "target_source_id": str(target_sid),
            "predicted_position": None if pred_idx is None else int(pred_idx),
            "exact_id_success": float(attack.metrics.get("exact_identification_success", np.nan)),
            "prior_rank": float(attack.metrics.get("prior_rank", np.nan)),
            "posterior_true_probability": float(attack.metrics.get("posterior_true_probability", np.nan)),
            "posterior_top1_probability": float(attack.metrics.get("posterior_top1_probability", np.nan)),
            "posterior_effective_candidates": float(attack.metrics.get("posterior_effective_candidates", np.nan)),
            "instance_bound_finite_opt": float(diagnostics.get("bound_direct_instance_finite_opt", np.nan)),
            "instance_bound_center_max": float(diagnostics.get("bound_direct_instance_center_max", np.nan)),
            "model_pairwise_snr_median": float(diagnostics.get("model_pairwise_snr_median", np.nan)),
            "model_nn_snr_median": float(diagnostics.get("model_nn_snr_median", np.nan)),
            "sigma": float(release.privacy.ball.sigma),
            "accuracy": float(release.utility_metrics.get("accuracy", np.nan)),
        })

attack_df = pd.DataFrame(attack_rows)
display(attack_df)


In [ ]:
attack_summary = (
    attack_df.groupby("mechanism", as_index=False)
    .agg(
        empirical_exact_id=("exact_id_success", "mean"),
        mean_rank=("prior_rank", "mean"),
        posterior_effective_candidates=("posterior_effective_candidates", "mean"),
        instance_bound_finite_opt=("instance_bound_finite_opt", "mean"),
        model_pairwise_snr_median=("model_pairwise_snr_median", "mean"),
        sigma=("sigma", "mean"),
        accuracy=("accuracy", "mean"),
    )
)
attack_summary["oblivious_baseline"] = 1.0 / M
display(attack_summary)

x = np.arange(len(attack_summary))
width = 0.25
plt.bar(x - width, attack_summary["empirical_exact_id"], width=width, label="empirical exact-ID")
plt.bar(x, attack_summary["instance_bound_finite_opt"], width=width, label="instance finite bound")
plt.bar(x + width, attack_summary["oblivious_baseline"], width=width, label="baseline 1/m")
plt.xticks(x, attack_summary["mechanism"])
plt.ylim(0, 1.02)
plt.ylabel("Probability")
plt.title("Exact-ID attack: empirical success vs bound")
plt.legend()
plt.show()


## Relation to the official scripts

This notebook demonstrates the API path for one dataset and one representative configuration. The paper-scale scripts add grids over \(arepsilon\), \(m\), radius, seeds, anchors, support draws, and datasets, then write publication tables and figures.
